# Decision Trees & Random Forest: Data Cleansing

## Section 1: Spot the Dirty Data

In [1]:
import numpy as np
import pandas as pd

# Create sample dataset with missing values, inconsistent entries, and outliers
data = {
    'Passenger': [1, 2, 3, 4],
    'Sex': ['male', 'Female', 'M', 'male'],
    'Age': [np.nan, 25.0, 8.0, 35.0],
    'Fare': [512.0, 71.0, 21.0, np.nan],
    'Survived': [0, 1, 1, 0]
}
df = pd.DataFrame(data)
print("Initial Dirty Dataset:")
print(df)

Initial Dirty Dataset:
   Passenger     Sex   Age   Fare  Survived
0          1    male   NaN  512.0         0
1          2  Female  25.0   71.0         1
2          3       M   8.0   21.0         1
3          4    male  35.0    NaN         0


## Section 2: Tree-Based Cleansing

In [2]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

# 1. Standardize Sex labels
df['Sex'] = df['Sex'].replace({'M': 'male', 'Female': 'female'}).str.lower()

# Encode Sex for decision tree imputation
df_encoded = df.copy()
df_encoded['Sex_encoded'] = df_encoded['Sex'].map({'male': 0, 'female': 1})

# 2. Impute missing Ages using a Decision Tree Regressor
train_age = df_encoded[df_encoded['Age'].notna()]
test_age = df_encoded[df_encoded['Age'].isna()]

X_train_age = train_age[['Sex_encoded', 'Survived']]
y_train_age = train_age['Age']
X_test_age = test_age[['Sex_encoded', 'Survived']]

dt_age = DecisionTreeRegressor(random_state=42)
dt_age.fit(X_train_age, y_train_age)
df.loc[df['Age'].isna(), 'Age'] = dt_age.predict(X_test_age)

# Update encoded df with imputed Age
df_encoded['Age'] = df['Age']

# 3. Impute missing Fares using a Decision Tree Regressor
train_fare = df_encoded[df_encoded['Fare'].notna()]
test_fare = df_encoded[df_encoded['Fare'].isna()]

X_train_fare = train_fare[['Sex_encoded', 'Age', 'Survived']]
y_train_fare = train_fare['Fare']
X_test_fare = test_fare[['Sex_encoded', 'Age', 'Survived']]

dt_fare = DecisionTreeRegressor(random_state=42)
dt_fare.fit(X_train_fare, y_train_fare)
df.loc[df['Fare'].isna(), 'Fare'] = dt_fare.predict(X_test_fare)

# 4. Flag Outliers (Fare > 300)
df['Is_Outlier'] = df['Fare'] > 300

print("\nCleaned and Processed Dataset:")
print(df)


Cleaned and Processed Dataset:
   Passenger     Sex   Age   Fare  Survived  Is_Outlier
0          1    male  35.0  512.0         0        True
1          2  female  25.0   71.0         1       False
2          3    male   8.0   21.0         1       False
3          4    male  35.0  512.0         0        True


## Section 3: Random Forest Robustness

In [3]:
from sklearn.ensemble import RandomForestClassifier

# Prepare data for training
df_clean_encoded = df.copy()
df_clean_encoded['Sex'] = df_clean_encoded['Sex'].map({'male': 0, 'female': 1})

X = df_clean_encoded[['Sex', 'Age', 'Fare']]
y = df_clean_encoded['Survived']

# Train Random Forest Classifier to demonstrate handling of noisy data/outliers
rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X, y)

print("\nRandom Forest trained successfully on cleaned data.")


Random Forest trained successfully on cleaned data.
